# BANKING77: from LLM scores to routing decisions

Notebook 02 showed how Qwen scores the 77 possible intents. Here we use
those scores to decide **when to answer automatically and when to defer
a request to human review**.

We compare two rules: LAC, which builds calibrated prediction sets and
automates only one-label sets, and naive thresholding, which accepts the
highest-scoring intent when its weight exceeds a cutoff.

The experiment uses 200 calibration requests and 200 separate evaluation
requests from BANKING77's official training data. We explore how changing
each rule's settings affects coverage, automation and errors. This is a
development study, not an evaluation on the official test set.

In [2]:
from pathlib import Path
from time import perf_counter

import mlx.core as mx
import numpy as np
from mlx_lm import load

from conformal_selective_prediction.calibration import conformal_quantile
from conformal_selective_prediction.data import load_banking77
from conformal_selective_prediction.metrics import (
    automated_error_rate,
    automation_rate,
    average_set_size,
    empirical_coverage,
)
from conformal_selective_prediction.prediction_sets import lac_prediction_sets
from conformal_selective_prediction.scores import lac_scores
from conformal_selective_prediction.selection import singleton_mask
from _scoring import score_candidates

## 1. Separate calibration from evaluation

We sample 400 distinct requests from the portion of BANKING77's official
training data reserved for calibration by the existing loader. The first
200 determine the LAC thresholds; the remaining 200 evaluate the routing
rules on different requests.

The random seed is fixed, and both methods use the same evaluation
requests. The official test set is not used in this notebook.

In [49]:
data = load_banking77(Path("../../data/raw/banking77"), random_seed=42)
class_names = data.class_names
n_calibration = 200
n_evaluation = 200

rng = np.random.default_rng(42)
sample_size = n_calibration + n_evaluation
sample_indices = rng.choice(
    len(data.calibration.texts), size=sample_size, replace=False
)
requests = data.calibration.texts[sample_indices]
labels = data.calibration.labels[sample_indices]

calibration_labels = labels[:n_calibration]
evaluation_labels = labels[n_calibration:]
evaluation_requests = requests[n_calibration:]

## 2. Keep the same model and prompt

We reuse Qwen, the prompt and summed token log-probabilities from notebook 02. Each candidate includes the end-of-response token. There is no training or prompt tuning here.

In [50]:
# Limit unused buffers on the 16 GB Mac.
mx.set_cache_limit(512 * 1024**2)

model, tokenizer = load(
    "mlx-community/Qwen3-4B-Instruct-2507-4bit",
    revision="50d427756c6b1b2fe0c0a10f67fbda1fc8e82c1b",
)

instructions = (
    "Classify the banking customer request.\n"
    "Choose one of these intents and return only its name:\n" + "\n".join(class_names)
)


# Format one request with the fixed list of possible intents.
def build_prompt(request: str) -> str:
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": request},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


candidate_ids = []
for label in class_names:
    label_ids = tokenizer.encode(label, add_special_tokens=False)
    candidate_ids.append(label_ids + [tokenizer.eos_token_id])

Fetching 11 files: 100%|█████████████████████| 11/11 [00:00<00:00, 1192.80it/s]


## 3. Obtain the candidate weights

The helper scores each request against all 77 intents. Softmax converts
the summed token log-probabilities into weights that sum to one. These
are **relative candidate weights, not calibrated probabilities of
correctness**: one intent can dominate the alternatives and still be wrong.

In [51]:
score_rows = []
start = perf_counter()

for request in requests:
    prompt = build_prompt(str(request))
    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
    scores = score_candidates(model, prompt_ids, candidate_ids)
    score_rows.append(scores)

log_scores = np.asarray(score_rows)
print(f"Scoring time: {(perf_counter() - start) / 60:.1f} minutes")

Scoring time: 55.8 minutes


In [52]:
row_maxima = log_scores.max(axis=1, keepdims=True)
shifted_scores = log_scores - row_maxima
unnormalized_weights = np.exp(shifted_scores)
row_totals = unnormalized_weights.sum(axis=1, keepdims=True)
weights = unnormalized_weights / row_totals

calibration_weights = weights[:n_calibration]
evaluation_weights = weights[n_calibration:]
predictions = evaluation_weights.argmax(axis=1)

## 4. Apply the two routing rules

- **LAC:** use the calibration examples to set a label-inclusion threshold
  for each coverage target: 80%, 70%, 60% and 50%. Automate only when exactly
  one intent remains; defer empty and multi-label sets.

- **Naive:** accept the highest-weight intent when its weight reaches
  0.90, 0.95, 0.99 or 0.999. Otherwise, defer the request.

A coverage target concerns keeping the correct intent in the prediction
set, not the accuracy of automatic decisions. Likewise, a naive cutoff
of 0.99 does not promise 99% accuracy.

If the weights and labels are still in the kernel, rerun from this section
to change the settings without repeating LLM scoring.

In [64]:
alphas = (0.20, 0.30, 0.40, 0.50)
calibration_scores = lac_scores(calibration_weights, calibration_labels)
lac_sets = {}

for alpha in alphas:
    threshold = conformal_quantile(calibration_scores, alpha)
    prediction_sets = lac_prediction_sets(evaluation_weights, threshold)
    lac_sets[alpha] = prediction_sets

In [65]:
confidence_cutoffs = (0.90, 0.95, 0.99, 0.999)
top_weights = evaluation_weights.max(axis=1)

## 5. Compare the settings

All eight configurations are evaluated on the same 200 requests.

- **Coverage** measures how often the prediction set contains the correct
  intent. **Set size** is the average number of retained intents across
  all evaluation requests.
- **Automation** is the fraction handled automatically. **Error on
  automated** is the fraction of those accepted decisions that are wrong,
  not the fraction of all requests.

The naive rule does not construct prediction sets, so its coverage and
set-size entries are left blank. An error of `N/A` would mean no requests
were automated. Displayed percentages are rounded.

In [66]:
print(
    f"{'Rule / setting':<22} {'Coverage':>10} {'Set size':>10} "
    f"{'Automation':>12} {'Error on automated':>20}"
)

for alpha, prediction_sets in lac_sets.items():
    mask = singleton_mask(prediction_sets)
    coverage = empirical_coverage(prediction_sets, evaluation_labels)
    mean_size = average_set_size(prediction_sets)
    rate = automation_rate(mask)
    error = automated_error_rate(predictions, evaluation_labels, mask)
    error_text = "N/A" if np.isnan(error) else f"{error:.0%}"
    name = f"LAC target {1 - alpha:.0%}"
    print(
        f"{name:<22} {coverage:>10.0%} {mean_size:>10.1f} "
        f"{rate:>12.0%} {error_text:>20}"
    )

for cutoff in confidence_cutoffs:
    mask = top_weights >= cutoff
    rate = automation_rate(mask)
    error = automated_error_rate(predictions, evaluation_labels, mask)
    error_text = "N/A" if np.isnan(error) else f"{error:.0%}"
    name = f"Naive cutoff {cutoff:.3f}"
    print(f"{name:<22} {'-':>10} {'-':>10} {rate:>12.0%} {error_text:>20}")

Rule / setting           Coverage   Set size   Automation   Error on automated
LAC target 80%                86%        6.5          20%                  13%
LAC target 70%                74%        2.7          50%                  21%
LAC target 60%                67%        1.5          75%                  30%
LAC target 50%                55%        0.8          82%                  33%
Naive cutoff 0.900              -          -          86%                  35%
Naive cutoff 0.950              -          -          84%                  35%
Naive cutoff 0.990              -          -          77%                  32%
Naive cutoff 0.999              -          -          68%                  28%


## 6. Look at three decisions

We inspect the first three evaluation requests using LAC's 80% coverage
target and a naive cutoff of 0.99.

These settings automate very different proportions of requests: 20% for
LAC and 77% for naive. The examples illustrate why a rule accepts or defers
a request; they are not a comparison at matched automation.

Both rules start from the same highest-weight intent. LAC's decision
depends on how many intents survive its threshold, while naive considers
only the largest weight.

In [67]:
example_alpha = 0.20
example_cutoff = 0.99

prediction_sets = lac_sets[example_alpha]
lac_automated = singleton_mask(prediction_sets)
set_sizes = prediction_sets.sum(axis=1)
naive_automated = top_weights >= example_cutoff

for index in range(3):
    lac_action = "automate" if lac_automated[index] else "defer"
    naive_action = "automate" if naive_automated[index] else "defer"

    print("Request:", evaluation_requests[index])
    print("True intent:", class_names[evaluation_labels[index]])
    print("Top intent:", class_names[predictions[index]])
    print(f"LAC: {lac_action}, set size = {set_sizes[index]}")
    print(f"Naive: {naive_action}, top weight = {top_weights[index]:.6f}")
    print()

Request: I was charged an extra dollar on my account and would like to know the reason.
True intent: extra_charge_on_statement
Top intent: extra_charge_on_statement
LAC: automate, set size = 1
Naive: automate, top weight = 1.000000

Request: How much transactions do I get with a disposable card?
True intent: disposable_card_limits
Top intent: disposable_card_limits
LAC: defer, set size = 2
Naive: automate, top weight = 1.000000

Request: How do I set up an account for my children?
True intent: age_limit
Top intent: age_limit
LAC: defer, set size = 4
Naive: automate, top weight = 0.999999



## What this run shows

- **LAC achieves the coverage targets shown, but coverage comes with a
  review burden.** At the 80% target, observed coverage is 86%, with an
  average of 6.5 intents per set. Only 20% of requests are automated, and
  13% of those decisions are wrong. Retaining the correct answer is
  therefore different from making reliable automatic decisions.

- **More automation comes at the expense of protection and accuracy.**
  Lowering the LAC target from 80% to 50% increases automation from 20%
  to 82%, while automated-case error rises from 13% to 33%. At the lowest
  target, average set size falls to 0.8, which also implies that some
  requests receive empty sets and are deferred.

- **Very high candidate weights still accompany substantial errors.**
  Raising the naive cutoff from 0.90 to 0.999 reduces automation from 86%
  to 68% and error from 35% to 28%. Even the strictest cutoff tested is
  not a reliable certificate that the chosen intent is correct.

- **These results do not establish a clear automation–error advantage
  for LAC.** At nearby operating points, LAC gives 75% automation with
  30% error, versus 77% automation with 32% error for naive. These are
  similar, not identical, results. LAC also reaches more selective settings,
  but the tested naive cutoffs do not cover those lower automation rates,
  so that region has not been compared fairly.

The main conclusion is that LAC makes the coverage–automation trade-off
explicit, but does not remove the model's classification errors. Both
methods still accept substantial mistakes at higher automation rates.
This single split supports an exploratory comparison, not a deployment
recommendation or a claim that the methods are equivalent.